In [1]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")
print("Split tersedia:", list(dataset.keys()))

split_name = "train" if "train" in dataset else list(dataset.keys())[0]
df = pd.DataFrame(dataset[split_name])

print("\n=== INFORMASI DATASET ===")
print(f"Jumlah data (baris): {len(df)}")
print(f"Jumlah kolom: {df.shape[1]}")
print(f"Nama kolom: {list(df.columns)}")

print("\n5 baris pertama:")
display(df.head())

for col in df.columns:
    if any(k in col.lower() for k in ["label", "sentiment", "sentimen"]):
        print(f"\nDistribusi label ({col}):")
        print(df[col].value_counts(dropna=False))

c:\Users\cinnn\Desktop\kuliah\pak riski\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\cinnn\Desktop\kuliah\pak riski\NLP\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\cinnn\.cache\huggingface\hub\datasets--dipawidia--ecommerce-product-reviews-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run

Split tersedia: ['train']

=== INFORMASI DATASET ===
Jumlah data (baris): 11606
Jumlah kolom: 3
Nama kolom: ['review', 'translate', 'sentimen']

5 baris pertama:


,review,translate,sentimen
0,Suara dan mic puas. Hanya saya pesan yg nyala ...,sound and mic satisfied. I only ordered the on...,0
1,Good. Mudah-mudahan awet. Makasih,good. hopefully durable. Thanks,1
2,"kondisi dus robek2, tolong di perhatikan lagi","the condition of the box is torn, please pay a...",0
3,Kualitas produk baik. Harga terjangkau dan pen...,product quality is good. affordable prices and...,0
4,"Overall bagus, tombol berfungsi semuanya, tapi...","overall is good, the buttons all work, but the...",0



Distribusi label (sentimen):
sentimen
0    5803
1    5803
Name: count, dtype: int64


In [2]:
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm import tqdm

tqdm.pandas()

stop_factory = StopWordRemoverFactory()
stopwords_id = set(stop_factory.get_stop_words())

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()


def case_folding(text: str) -> str:
    return str(text).lower()


def cleaning(text: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # hapus URL
    text = re.sub(r"\d+", " ", text)  # hapus angka
    text = text.translate(str.maketrans("", "", string.punctuation))  # hapus tanda baca
    text = re.sub(r"[^a-z\s]", " ", text)  # hapus karakter khusus
    text = re.sub(r"\s+", " ", text).strip()  # rapikan spasi
    return text


def tokenizing(text: str) -> list[str]:
    return text.split()


def stopword_removal(tokens: list[str]) -> list[str]:
    return [word for word in tokens if word not in stopwords_id]


def stemming(tokens: list[str]) -> str:
    stemmed_tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(stemmed_tokens)


def preprocess_pipeline(text: str) -> str:
    text = case_folding(text)
    text = cleaning(text)
    tokens = tokenizing(text)
    tokens = stopword_removal(tokens)
    return stemming(tokens)


print("Pipeline preprocessing siap.")

Pipeline preprocessing siap.


In [3]:
# Cek nama kolom ulasan di output Cell 1; jika bukan 'review'/'text', ganti sesuai nama kolomnya
kandidat_kolom = ["review", "text", "ulasan", "content"]
kolom_teks = next((c for c in kandidat_kolom if c in df.columns), df.columns[0])
print(f"Memproses kolom teks: '{kolom_teks}'")

df["teks_bersih"] = df[kolom_teks].progress_apply(preprocess_pipeline)

pd.set_option("display.max_colwidth", None)
contoh_10 = df[[kolom_teks, "teks_bersih"]].head(10).copy()
contoh_10.columns = ["Sebelum Preprocessing", "Sesudah Preprocessing"]
display(contoh_10)

Memproses kolom teks: 'review'


100%|██████████| 11606/11606 [20:40<00:00,  9.36it/s]


,Sebelum Preprocessing,Sesudah Preprocessing
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datang ga nyala,suara mic puas pesan yg nyala sy pasti lg klik yg nyala tp yg datang ga nyala
1,Good. Mudah-mudahan awet. Makasih,good mudahmudahan awet makasih
2,"kondisi dus robek2, tolong di perhatikan lagi",kondisi dus robek perhati
3,Kualitas produk baik. Harga terjangkau dan pengirimannya cepet banget. Udh 3x order disini oke lah tp sayangnya klo buat kado gabisa nitip pake kartu ucapan atau dibungkus pake kertas kado :(,kualitas produk baik harga jangkau kirim cepet banget udh x order sini oke lah tp sayang klo buat kado gabisa nitip pake kartu ucap bungkus pake kertas kado
4,"Overall bagus, tombol berfungsi semuanya, tapi packaging kurang karena sudah penyot dan robek sedikit",overall bagus tombol fungsi semua packaging kurang penyot robek sedikit
5,"Produk ini original, Dan kualitasnya sangat baik. kecepatan pengiriman sanagt cepat, recommended seller.",produk original kualitas sangat baik cepat kirim sanagt cepat recommended seller
6,kualitas sangat jelek. bocor,kualitas sangat jelek bocor
7,"terimakasih banyak ka ya, barang sudah diterima dengan baik. sukses terus ka",terimakasih banyak ka barang terima baik sukses terus ka
8,"Body nya lebih berat dari cm 1579, spiral nya lebih tipis dibanding cm 1579, semoga awet, beli CM 1659 penasaran Aja, yang cm 1579 udah mau rusak mesin nya, Harga lebih murah 4k dari cm 1579 Fast delivery, courier Ramah sangat, ty seller",body nya lebih berat cm spiral nya lebih tipis banding cm moga awet beli cm penasaran aja cm udah mau rusak mesin nya harga lebih murah k cm fast delivery courier ramah sangat ty seller
9,pesanan di terima tanggal 27 January 2021 tetapi baru bisa saya buka pagi ini tanggal 29 January 2021 pukul 06.00 dan ternyata tab SKETSA ADVAN tidak bisa di nyalakan sama sekali dan di cas juga posisi Tab tidak bisa menyala,pesan terima tanggal january baru buka pagi tanggal january pukul nyata tab sketsa advan nyala sama sekali cas posisi tab nyala
